#### The Retail UCI Dataset

In [152]:
import pandas as pd
path = 'data/online_retail.xlsx'
df = pd.read_excel(path)

In [153]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


- `InvoiceNo`: a 6-digit integral number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation.

- `StockCode`: a 5-digit integral number uniquely assigned to each distinct products.

- `Description`: product name.

- `Quantity`: The quantities of each product per transaction.

- `InvoiceDate`: the day and time when each transaction was generated.

- `UnitPrice`: product price per unit.

- `CustomerID`: 5-digit integral number uniquely assigned to each customer.

- `Country`: the name of the country where each customer resides.

In [154]:
df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

In [155]:
df.shape

(541909, 8)

In [156]:
n = 9288 # number of cancel InvoiceNo
df_a = df[df['InvoiceNo'].astype(str).str.startswith('C', na=False)]

In [157]:
df_b = df[df['Quantity'] < 0]

In [158]:
df_diff = (
    df_b.merge(df_a.drop_duplicates(), how='left', indicator=True)
    .query('_merge == "left_only"')
    .drop(columns='_merge')
)

In [159]:
df_diff

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
26,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
38,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
85,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
86,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
87,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
10545,581210,23395,check,-26,2011-12-07 18:36:00,0.0,NaN,United Kingdom
10546,581212,22578,lost,-1050,2011-12-07 18:38:00,0.0,NaN,United Kingdom
10547,581213,22576,check,-30,2011-12-07 18:38:00,0.0,NaN,United Kingdom
10548,581226,23090,missing,-338,2011-12-08 09:56:00,0.0,NaN,United Kingdom


In [160]:
col = ['InvoiceNo', 'StockCode', 'Description', 'CustomerID']
df[col].nunique()

InvoiceNo      25900
StockCode       4070
Description     4223
CustomerID      4372
dtype: int64

In [161]:
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

- There are more than 135080/541909 records that do not have the id of customer and some invoices do not contains product's description. These records should be deeply explore in the next part of this notebook.

- The number of unique `InvoiceNo` << the numbers of rows in UCI datasets. In addition, the `CustomerID` unique number exists in this dataset is less than `InvoiceNo`. We can conclude that a invoice could contain more than one customer.

- `Quantity` column has a significant number of "errorneous" observers, which > 10k records have the negative quantity value. Especially, 90% of these rows have cancelled invoice. This problem will be analysed in the next step of EDA.

In [162]:
df[['UnitPrice', 'Quantity']].describe()

,UnitPrice,Quantity
count,541909.000000,541909.000000
mean,4.611114,9.552250
std,96.759853,218.081158
min,-11062.060000,-80995.000000
25%,1.250000,1.000000
50%,2.080000,3.000000
75%,4.130000,10.000000
max,38970.000000,80995.000000


In [163]:
df[df['Description'] == 'Adjust bad debt']

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


Records with the description "Adjust bad debt" contain anomalous UnitPrice values, including unusually large positive and negative amounts. These records represent accounting adjustments rather than actual retail transactions and may distort the Monetary component of RFM analysis.

Therefore, these records are excluded from the dataset used for customer segmentation, as they do not reflect genuine purchasing behavior and have missing CustomerID values that prevent reliable customer-level attribution. The original dataset is preserved for traceability.

In [164]:
df = df[df['Description'] != 'Adjust bad debt']

##### `StockCode`

In [165]:
# StockCode có đúng 5 chữ số
is_5_digit = df['StockCode'].astype(str).str.match(r'^\d{5}$')

# StockCode có 5 chữ số kèm theo 1 số chữ cái
is_variant_product = df['StockCode'].astype(str).str.match(r'^\d{5}[a-zA-Z]{1,2}$')

variants = df[is_variant_product]
print("Số mã biến thể (5 số + chữ cái):", variants['StockCode'].nunique())

# 2. Các mã dịch vụ / dị biệt
special_codes = df[~(is_5_digit | is_variant_product)]
print("Các mã dịch vụ đặc biệt:")
print(special_codes[['StockCode', 'Description']].drop_duplicates())

Số mã biến thể (5 số + chữ cái): 1091
Các mã dịch vụ đặc biệt:
           StockCode                          Description
45              POST                              POSTAGE
141                D                             Discount
1423              C2                             CARRIAGE
1814             DOT                       DOTCOM POSTAGE
2239               M                               Manual
4406    BANK CHARGES                         Bank Charges
14436              S                              SAMPLES
14514      AMAZONFEE                           AMAZON FEE
21326       DCGS0076         SUNJAR LED NIGHT NIGHT LIGHT
24906       DCGS0003                  BOXED GLASS ASHTRAY
38248   gift_0001_40   Dotcomgiftshop Gift Voucher £40.00
40052       DCGS0070                CAMOUFLAGE DOG COLLAR
40383              m                               Manual
42057   gift_0001_50   Dotcomgiftshop Gift Voucher £50.00
44725   gift_0001_30   Dotcomgiftshop Gift Voucher £30.00
44794   g

- Standard & Variant Products: 5 digits (e.g., `22423`) or 5 digits with letter suffixes (e.g., `85123A`) representing color/size variants $\rightarrow$ **Keep**.
- Non-Product Codes: Service, fee, or manual entries (e.g., `POST`, `DOT`, `C2`, `D`, `M`, `BANK CHARGES`, `AMAZONFEE`, `gift_...`, `B`)
    + Shipping (`POST`), platform fees, and discounts represent operational overhead, not customer merchandise demand.
    + Internal accounting adjustments (`B`, `M`) and heavy fees skew `Monetary` value and inflate `Frequency`.
    + Prevents Algorithm Bias: Prevents postage (`POST`) from being falsely flagged as a "top-selling product" in basket analysis/clustering.
    + Gift cards (`gift_...`) are payment instruments, not consumable goods.
- These stock codes will eliminated from the dataset. With the group of stock codes like `DCGS...` and `PAD` should be check carefully to determine that whether this group indicates a valid product from customers.

In [166]:
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135077
Country             0
dtype: int64

In [167]:
special_codes[
    (
        special_codes['StockCode']
        .astype(str) 
        .str.startswith("DCGS")
    ) | 
    (
        special_codes['StockCode']
        .astype(str)
        .str.startswith('PADS')
    )
]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
21326,538071,DCGS0076,SUNJAR LED NIGHT NIGHT LIGHT,1,2010-12-09 14:09:00,16.130,NaN,United Kingdom
24906,538349,DCGS0003,BOXED GLASS ASHTRAY,1,2010-12-10 14:59:00,2.510,NaN,United Kingdom
36460,539451,DCGS0003,BOXED GLASS ASHTRAY,1,2010-12-17 16:59:00,2.510,NaN,United Kingdom
39313,539631,DCGS0076,SUNJAR LED NIGHT NIGHT LIGHT,2,2010-12-20 15:03:00,16.130,NaN,United Kingdom
40052,539718,DCGS0070,CAMOUFLAGE DOG COLLAR,1,2010-12-21 13:06:00,12.720,NaN,United Kingdom
74825,542529,DCGS0055,NaN,-1,2011-01-28 13:08:00,0.000,NaN,United Kingdom
74838,542531,DCGS0072,NaN,-1,2011-01-28 13:08:00,0.000,NaN,United Kingdom
74839,542532,DCGS0074,NaN,-1,2011-01-28 13:09:00,0.000,NaN,United Kingdom
75053,542541,DCGS0069,OOH LA LA DOGS COLLAR,1,2011-01-28 14:25:00,15.790,NaN,United Kingdom
75295,542582,DCGS0057,NaN,-6,2011-01-28 15:01:00,0.000,NaN,United Kingdom


In [168]:
with pd.option_context('display.max_rows', None):
    display(df['Description'].value_counts())

Description
WHITE HANGING HEART T-LIGHT HOLDER     2369
REGENCY CAKESTAND 3 TIER               2200
JUMBO BAG RED RETROSPOT                2159
PARTY BUNTING                          1727
LUNCH BAG RED RETROSPOT                1638
ASSORTED COLOUR BIRD ORNAMENT          1501
SET OF 3 CAKE TINS PANTRY DESIGN       1473
PACK OF 72 RETROSPOT CAKE CASES        1385
LUNCH BAG  BLACK SKULL.                1350
NATURAL SLATE HEART CHALKBOARD         1280
POSTAGE                                1252
JUMBO BAG PINK POLKADOT                1251
HEART OF WICKER SMALL                  1237
JAM MAKING SET WITH JARS               1229
JUMBO STORAGE BAG SUKI                 1214
PAPER CHAIN KIT 50'S CHRISTMAS         1210
JUMBO SHOPPER VINTAGE RED PAISLEY      1202
LUNCH BAG CARS BLUE                    1197
LUNCH BAG SPACEBOY DESIGN              1192
JAM MAKING SET PRINTED                 1182
RECIPE BOX PANTRY YELLOW DESIGN        1180
SPOTTY BUNTING                         1172
LUNCH BAG SUKI DESIG

In [169]:
with pd.option_context('display.max_rows', None):
    display(df[df['UnitPrice'] == 0].loc[:, 'Description'].value_counts())

Description
check                                  159
?                                       47
damages                                 45
damaged                                 43
found                                   25
sold as set on dotcom                   20
adjustment                              16
Damaged                                 14
Unsaleable, destroyed.                   9
thrown away                              9
FRENCH BLUE METAL DOOR SIGN 1            9
amazon                                   8
FRENCH BLUE METAL DOOR SIGN 8            8
Found                                    8
FRENCH BLUE METAL DOOR SIGN 4            7
RECIPE BOX PANTRY YELLOW DESIGN          7
Amazon                                   7
FRENCH BLUE METAL DOOR SIGN No           7
??                                       7
OWL DOORSTOP                             7
FRENCH BLUE METAL DOOR SIGN 3            7
MINT KITCHEN SCALES                      6
FRENCH BLUE METAL DOOR SIGN 7            6

- Lowercase & Symbol Descriptions (`check`, `damaged`, `found`, `?`): Standard product names in the catalog are strictly UPPERCASE. Lowercase descriptions and question marks represent internal warehouse memos (e.g., stock checks, damaged/unsaleable goods, inventory adjustments) rather than customer purchases.
- 100% Overlap with `UnitPrice == 0`: Every single lowercase/memo description occurs exclusively when `UnitPrice == 0` (e.g., all 159 `check` and 47 `?` records have a price of `0`).
- Uppercase Products with `UnitPrice == 0`: Represent either unannotated stock adjustments (`CustomerID` is `NaN`) or free promotional/replacement items (`CustomerID` is present), neither of which contribute to revenue.
- The `StockCode` with value `PADS` or starts with `DCGS...` appears with missing `CustomerID`.

In [170]:
df = df[(df['UnitPrice'] != 0) & (is_5_digit | is_variant_product)]

##### Missing value handle

Since records with missing `CustomerID` values cannot be reliably linked to individual customers for subsequent profiling and tracking, these records were excluded from the dataset.

In [172]:
# Dataset contains only missing CustomerID
df_clean = df[~df['CustomerID'].isna()]

##### Negative Quanity and Cancelled Invoice

In [175]:
df_clean[(df_clean['InvoiceNo'].astype(str).str.startswith('C')) & (df_clean['Quantity'] < 0)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
...,...,...,...,...,...,...,...,...
540448,C581490,22178,VICTORIAN GLASS HANGING T-LIGHT,-12,2011-12-09 09:57:00,1.95,14397.0,United Kingdom
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


In [174]:
df_clean[df_clean['Quantity'] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
...,...,...,...,...,...,...,...,...
540448,C581490,22178,VICTORIAN GLASS HANGING T-LIGHT,-12,2011-12-09 09:57:00,1.95,14397.0,United Kingdom
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


##### Cleaning Data Decision

- Remove all records with missing values in `CustomerID`.
- Remove records where `UnitPrice` is zero or negative.
- Remove records whose `Description` is not written entirely in uppercase letters.
- Treat invoices with an `InvoiceNo` prefixed by `C` and a negative `Quantity` as potential indicators of customer return behavior. These two features can be used together to identify transactions associated with product returns or cancellations.